<a href="https://colab.research.google.com/github/cpython-projects/python_da_17_03_26/blob/main/lesson_21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files

import warnings
warnings.filterwarnings("ignore")

# **1. Що таке сезонність**

## 1.1 Визначення

Коли ми говоримо про сезонність, ми говоримо **не про календарні пори року**, а про повторювані цикли в даних. Це може бути що завгодно: година, день, тиждень, місяць, квартал або навіть 10-хвилинні інтервали — якщо в них є стійкий патерн.

**Сезонність** — це повторюваний і стійкий патерн поведінки часовго ряду, який виникає з фіксованою періодичністю.
Тобто — це щось, що повторюється *регулярно та передбачувано*.

**Часовий ряд (time series)** — це набір значень, упорядкованих у часі, зазвичай з фіксованою частотою (година, день, тиждень, місяць).
Найголовніша властивість: порядок часу має значення, і переставляти рядки не можна.

Сезонність — це одна з ключових речей, без яких **неможливо робити правильний аналіз даних**.

**Сезонність допомагає**:

1. **Прогнозувати завантаження та попит**
   Приклад: у енергетичного оператора щодня о 19:00 фіксується пік споживання. Це стабільно з року в рік → отже, можна планувати генерацію потужностей.

2. **Знаходити аномалії**
   Якщо зазвичай у суботу трафік у 1.8–2.2 раза вищий, а сьогодні він раптово нижчий → це привід для розслідування.
   TSA Air Travel Data (США) показують колосальний спад польотів 25 грудня щороку → якщо в один рік різкого спаду немає — це аномалія.

3. **Оптимізувати бюджети**
   Ритейлер Walmart демонструє пік продажів у листопаді–грудні (Black Friday → Christmas). Якщо маркетолог «забуде» про сезонність, він просто зіллє бюджет у жовтні.

4. **Покращувати прогнозні моделі**
   Моделі ARIMA, Prophet і навіть лінійна регресія потребують **dummy-змінних за місяцями / днями тижня**.

## 1.2 Які типи сезонності бувають і як їх відрізняти

Сезонність буває кількох видів. Найважливіший критерій — **період повторення патерна**.
У Pandas тип сезонності можна відрізнити за групуванням за часом, а на графіках — за повторюваними піками / провалами.

---

### 1. Щоденна (внутрішньодобова) сезонність

Період: 24 години
Повторювані патерни всередині дня.

#### Приклади

* Споживання електроенергії PJM: **пік увечері (18–20 годин), мінімум уночі (3–5 годин)**.
* Трафік сайту: піки вранці та ввечері, провал глибокої ночі.
* Поїздки Uber: пік уночі в п’ятницю / суботу, провал зранку.

**Ознака:** на графіку чіткий «хвилеподібний» цикл у межах доби.

---

### 2. Тижнева сезонність

Період: 7 днів
Повторення за днями тижня.

#### Приклади

* Трафік сайту: **будні < вихідні** (коли люди вдома).
* Відвідуваність спортзалів: **понеділок — найзавантаженіший день**, субота — спад.
* Пасажиропотік TSA: субота — мінімум.

**Ознака:** одні дні тижня стабільно високі, інші — стабільно низькі.

---

### 3. Місячна сезонність

Період: 12 місяців
Повторення: щороку ті самі місяці повторюються з піками та провалами.

#### Приклади

* Продажі Walmart: пік у листопаді–грудні.
* Споживання газу: пік узимку.
* Туризм: пік улітку.

**Ознака:** на графіку одні місяці завжди вищі за інші.

---

### 4. Річна сезонність

Період: 1 рік
Повторення за роками на великих інтервалах.

Це рідкісний випадок у звичайній бізнес-аналітиці, але трапляється при:

* багаторічній статистиці погоди
* макроекономічних даних
* сільському господарстві (врожайність)

**Ознака:** дані коливаються з року в рік не випадково, а повторюються за циклами.

---

### 5. «Нестандартна» сезонність (цикл ≠ календарю)

Період визначається не датою, а бізнес-процесом.

#### Приклади

* Call-центр: зміни по 6 годин → цикл = 6 годин.
* Виробництво: 4 зміни → цикл = 8 годин.
* Мобільна гра: івенти щоп’ятниці → цикл = 7 днів.
* Завантаження кур’єрської служби: піки перед 18:00, коли зачиняються магазини.

**Ознака:** дані «дихають» у межах внутрішніх циклів компанії.

---

### 6. Множинна сезонність

Період: кілька циклів одночасно.

#### Приклади

* Електроенергія: внутрішньодобова + сезон року.
* Трафік сайту: щоденна + тижнева.
* Продажі: місячна + внутрішньотижнева.

Тобто в даних є кілька патернів:
📉 щодня — своя міні-хвиля
📉 щотижня — є неділя
📈 кожного грудня — пік продажів

**Ознака:** графік виглядає як «хвиля всередині хвилі».

---

## Головний критерій: повторюваність

Сезонність — це не просто «вчора був пік, сьогодні — провал».

Вона повинна:

* повторюватися
* бути регулярною
* мати фіксований період

І лише тоді це сезонність, а не шум чи тренд.

# **2. Підготовка часових даних у Pandas**


In [ ]:
data = [
    {"date": "2024/01/01", "value": 103.1},
    {"date": "2024/01/02", "value": 100.4},
    {"date": "2024/01/04", "value": 111.8},  # 01-03 пропущено
    {"date": "2024/01/05", "value": 99.7},
    {"date": "2024/01/06", "value": 95.1},
    {"date": "2024/01/07", "value": 104.3},
    {"date": "01-15-2024", "value": 98.5},   # інший формат
    {"date": "2024/01/08", "value": 105.2},
    {"date": "2024/01/08", "value": 105.2},  # дублікат
    {"date": "2024/01/09", "value": 110.1},
    {"date": "2024.02.03", "value": 120.7},  # інший формат
    {"date": "2024/02/04", "value": 98.2},
    {"date": "2024/02/05", "value": 500.0},  # викид
    {"date": "2024/02/06", "value": 102.6},
    {"date": "2024/02/07", "value": 96.2},
    {"date": "2024/02/08", "value": 101.5},
    {"date": "2024/02/09", "value": 105.8},
    {"date": "2024/02/10", "value": 400.0},  # ще один викид
    {"date": "2024/02/12", "value": 99.4},   # 02-11 пропущено
]

df = pd.DataFrame(data)
df

,date,value
0,2024/01/01,103.1
1,2024/01/02,100.4
2,2024/01/04,111.8
3,2024/01/05,99.7
4,2024/01/06,95.1
5,2024/01/07,104.3
6,01-15-2024,98.5
7,2024/01/08,105.2
8,2024/01/08,105.2
9,2024/01/09,110.1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    19 non-null     object 
 1   value   19 non-null     float64
dtypes: float64(1), object(1)
memory usage: 436.0+ bytes


Робота з сезонністю **завжди починається** з підготовки часового ряду.


## 2.1. Приведення дати

У часових рядах ключове — зробити так, щоб стовпець дати став:

1. **датою** (`datetime64`)
2. **індексом**
3. **відсортованим**


### Коли буває погано:

| Проблема                                  | Як вирішувати                                |
| ----------------------------------------- | -------------------------------------------- |
| різні формати: `2024/01/05`, `05-01-2024` | `pd.to_datetime(..., dayfirst=True)`         |
| помилки в датах                           | `errors="coerce"` → перетворює помилки в NaT |
| часові пояси                              | `tz_localize`, `tz_convert`                  |


## 2.2. Видалення дублікатів

У даних часового ряду **дублікати дат** можуть призвести до помилок під час:

* зміни частоти (`asfreq`)
* ресемплінгу (`resample`)
* аналізу сезонності

Навіть один повтор дати ламає розрахунки агрегатів і може спотворити графіки.


Видалити дублікати можна двома способами:

1️⃣ **Залишити перше значення**:

```python
df = df[~df.index.duplicated(keep='first')]
```

2️⃣ **Агрегувати значення дат, що дублюються** (наприклад, за середнім):

```python
df = df.groupby(df.index).mean()  # середнє для всіх числових стовпців усередині кожної групи
```


## 2.3. Перевірка частоти

Багато хто помилково вважає, що якщо дати йдуть по порядку — частота правильна.
Але в даних можуть бути:

* пропуски днів
* дублікати дат
* дивні інтервали (1, 1, 2, 1 дня…)

Перевірити частоту можна так:

```python
df.index.inferred_freq
```

### Якщо результат `None` → частота порушена

Для сезонності це критично: навіть один пропуск руйнує **тижневий патерн**.


In [ ]:
print(df.index.inferred_freq)

None


### **Приведення до рівномірної частоти**

Зазвичай обирають період, який має сенс для задачі:

* щоденна (`"D"`)
* погодинна (`"H"`)
* щомісячна (`"MS"` — початок місяця)

```python
df = df.asfreq("D")
```

### Що відбувається?

* пропущені дати → додаються
* значення → стають `NaN`

Таким чином ми отримуємо **чистий**, аналітично коректний ряд.


## 2.4. Очищення: пропуски та викиди

Після `asfreq()` майже завжди з’являються `NaN`.
Також у реальних даних можуть бути викиди: різкі піки або провали.

---

### **Заповнення пропусків**

Якщо дані метричні (продажі, трафік, погода):

```python
df["value"] = df["value"].interpolate()
```

Якщо дані дискретні (кількість замовлень):

```python
df["value"] = df["value"].fillna(method="ffill")
```

---

### **Видалення викидів**

Іноді у сезонності викид — це нормально (Чорна П’ятниця).
Але якщо немає знання домену → краще обережно обмежити:

```python
df["value"] = df["value"].clip(
    lower=df["value"].quantile(0.01),
    upper=df["value"].quantile(0.99)
)
```

### Що це робить?

* обрізає екстремальні 1% знизу/зверху
* *не видаляє точки*, просто коригує їх


# **3. Resample vs. GroupBy**

Перш ніж переходити до математичних методів на кшталт STL, потрібно побачити сезонність **очима**.
У 80% випадків візуального аналізу достатньо, щоб сказати: «Так, є тижневий патерн» або «Місячної сезонності немає».

---

Сезонність у часових рядах оцінюють двома різними методами:

1. **Resample** — зміна частоти часового ряду
2. **GroupBy** — агрегація за календарними категоріями (місяць, день тижня, година)

Ці методи **не виконують одне й те саме**.
Їх потрібно використовувати **для різних типів задач**.


## 3.1. Resample — для побудови рівномірного часового ряду

### Коли використовуємо `resample`?

* коли дані містять нерегулярні часові точки
* коли потрібно перевести ряд у **щомісячний / щотижневий / погодинний формат**
* коли потрібно побачити **сезонні цикли в реальному часі**
* коли важливі **конкретні дати**
* коли сезонність проявляється як хвилі на графіку

`Resample` відповідає на питання:

> “Як змінювалось значення в кожен місяць / тиждень у хронологічному порядку?”

---

### Яку агрегуючу функцію обирати?

| Тип даних                                        | Найкраща agg    | Пояснення                         |
| ------------------------------------------------ | --------------- | --------------------------------- |
| Продажі, замовлення, кількість подій             | **sum**         | сума за місяць = реальний обсяг   |
| Поведінкові метрики (конверсії, CTR, чек)        | **mean**        | середнє за місяць більш стабільне |
| Датасети з піками, аномаліями                    | **median**      | прибирає вплив викидів            |
| Технічні показники (час відповіді, навантаження) | mean або median | залежить від стабільності ряду    |


## 3.2. GroupBy — для виявлення повторюваного патерна

`GroupBy` використовується, коли ми хочемо знайти **типову поведінку за календарними категоріями**, наприклад:

* які місяці в середньому найсильніші?
* які дні тижня найактивніші?
* який профіль активності за годинами?

`GroupBy` **не створює часовий ряд**.
`GroupBy` створює **сезонний профіль**.

---

### **Що таке сезонний профіль?**

Це:

> **Середня форма поведінки метрики всередині періоду сезонності.**

Наприклад:

* всередині **року** → профіль з 12 точок (місяці)
* всередині **тижня** → профіль з 7 точок
* всередині **добу** → профіль з 24 точок
* всередині «день тижня × година» → теплова карта 7×24

Тобто сезонний профіль — це відповідь на питання:

> «Як виглядає типовий рік / типова неділя / типовий день?»

---

### Яку агрегуючу функцію обирати?

| Завдання                                | Найкраща agg | Пояснення                          |
| --------------------------------------- | ------------ | ---------------------------------- |
| Характерна середня поведінка місяця/дня | **mean**     | усереднює різні роки               |
| Якщо багато шуму / викидів              | **median**   | стійка до аномалій                 |
| Порівняння загального обсягу по місяцях | **sum**      | сума за всі роки по кожному місяцю |


## 3.3. Resample vs GroupBy — порівняння

| Питання                                  | Resample | GroupBy |
| ---------------------------------------- | -------- | ------- |
| Зберігається чи часовий порядок?         | ✔️ Так   | ❌ Ні    |
| Підходить для пошуку трендових хвиль?    | ✔️       | ❌       |
| Підходить для аналізу «типового місяця»? | ❌        | ✔️      |
| Агрегація за рівними проміжками часу     | ✔️       | ❌       |
| Агрегація за календарними категоріями    | ❌        | ✔️      |
| Для довгих рядів з нерівним кроком       | ✔️       | ❌       |
| Для побудови сезонного профілю           | ❌        | ✔️      |


## 3.4. Як правильно комбінувати Resample та GroupBy

### Крок 1 — Resample

Отримуємо рівномірний ряд і бачимо хвилі.

```python
monthly = df.resample('M').sum()
px.line(monthly)
```

### Крок 2 — GroupBy

Створюємо «портрет» сезонності.

```python
df["month"] = df.index.month
pattern = df.groupby("month")["value"].mean()
px.bar(pattern)
```

### Крок 3 — Порівнюємо

* якщо `resample` показує хвилі,
* і `groupby` показує закономірність за місяцями —

→ **це справжня сезонність**, а не випадковість.


# **4. Як відрізнити хибну сезонність**


Не все, що повторюється — це сезонність. Перевіряємо:

1. **Повторюваність**

   * Піки повинні бути **регулярними**, а не випадковими
2. **Стабільність амплітуди**

   * Якщо різниця в піках/провалах хаотична → швидше шум
3. **Контроль факторів**

   * Перевірте: чи немає зовнішньої події, що створює «псевдопатерн» (реклама, акція)
4. **Порівняння за кількома періодами**

   * Якщо субота одного разу пік, а на наступному тижні немає → хибна сезонність


# **5. Приклади**

## 5.1 Yellow Taxi Trips (NYC TLC) ([https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page))

Це реальні дані Нью-Йоркської таксомоторної комісії (TLC).
Кожен рядок — **одна поїздка** жовтого таксі.

---

### Службові поля

| Поле                   | Опис                                                                                         |
| ---------------------- | -------------------------------------------------------------------------------------------- |
| **VendorID**           | Ідентифікатор компанії-постачальника даних (зазвичай 1 або 2).                               |
| **RatecodeID**         | Тарифний код (звичайний, аеропортовий, negotiated fare тощо).                                |
| **store_and_fwd_flag** | Відкладена передача даних: `Y` — не було зв’язку, дані збережені в таксі і передані пізніше. |

---

### Дата та час

| Поле                      | Опис                                     |
| ------------------------- | ---------------------------------------- |
| **tpep_pickup_datetime**  | Час початку поїздки (момент посадки).    |
| **tpep_dropoff_datetime** | Час завершення поїздки (момент висадки). |

Використовуються для аналізу:

* добової сезонності
* тижневої сезонності
* піків попиту
* навантаження за годинами / днями тижня

---

### Пасажири

| Поле                | Опис                                                             |
| ------------------- | ---------------------------------------------------------------- |
| **passenger_count** | Скільки пасажирів було в машині (вказано водієм, іноді неточно). |

---

### Поїздка

| Поле              | Опис                                    |
| ----------------- | --------------------------------------- |
| **trip_distance** | Відстань поїздки в милях.               |
| **PULocationID**  | Ідентифікатор зони посадки (Taxi Zone). |
| **DOLocationID**  | Ідентифікатор зони висадки.             |

Зони беруться з NYC Taxi Zones shapefile.

---

### Вартість поїздки

| Поле                      | Опис                                   |
| ------------------------- | -------------------------------------- |
| **fare_amount**           | Базовий тариф (без додаткових зборів). |
| **extra**                 | Доплати: нічний тариф, «rush hour».    |
| **mta_tax**               | Податок MTA (0.50).                    |
| **tip_amount**            | Чаєві.                                 |
| **tolls_amount**          | Платні дороги.                         |
| **improvement_surcharge** | Фіксований збір 0.30.                  |
| **congestion_surcharge**  | Збір за завантаженість.                |
| **Airport_fee**           | Аеропортовий збір (JFK/LGA).           |
| **cbd_congestion_fee**    | Плата за центр Манхеттена.             |
| **total_amount**          | Підсумкова вартість поїздки.           |


In [ ]:
df = pd.read_parquet("https://github.com/cpython-projects/da_1709/raw/refs/heads/main/yellow_tripdata_2025-01.parquet")
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


Графік «зубчастий»: активність різко змінюється залежно від часу доби — це на графіку виглядає як повторювані хвилі. Але такий графік — це суміш даних за 30 днів. Він показує факт сезонності, але не її структуру.

Щоб зрозуміти, як саме поводиться система — коли починається пік, коли досягається максимум, коли спад, чим вихідні відрізняються від буднів — потрібен **профіль сезонності**.


### Добова сезонність


**1. Глибока нічна яма**

* З 1:00 до 5:00 кількість поїздок **різко падає**.
* Мінімум близько **5:00** — приблизно 600–700 поїздок.

Це очікувана поведінка для міського транспорту вночі.

**2. Ранковий ріст**

* З 5:00 починається **швидкий ріст**.
* В 7–9 ранку — **ранковий пік** (ефект commute).
* Commute = «щоденна дорога на роботу/навчання» (вранці і ввечері).

**3. Денне плато та пік**

* З 10:00 до 16:00 — стабільний ріст, активний день.
* Головний пік близько **17–18 годин** — максимум ~8.5–9 тис. поїздок.

Це вечірній "комм’ютинг + активний рух по місту".

**4. Спад ввечері**

* Після 19:00 тренд йде вниз.
* З 20:00 до 22:00 — все ще висока активність.
* До півночі — плавне зниження.

**Висновок:** Класична міська добова сезонність «U-подібна форма вночі + два піки комм’ютингу».


### Тижнева сезонність


**1. Робочі дні активніші за вихідні**

* Пн–Вт — близько **90–92 тис.** поїздок.
* Ср–Чт — **пік тижня**, до **120–122 тис.**

**2. П’ятниця тримається високо**

* Близько **117–118 тис.** — трохи нижче четверга, але все ще високий попит.

**3. Падіння у вихідні**

* Субота — помітний спад (**~95 тис.**).
* Неділя — трохи вище суботи (**~100 тис.**).

**Висновок:**

* Максимальна активність — **посередині робочого тижня**: бізнес-активність, комм’ютинг, логістика.
* Мінімум — **рано вранці** та **у вихідні**.


# **5. Seasonal Decomposition — розклад часових рядів за допомогою STL**


STL (Seasonal and Trend decomposition using Loess) — це метод розкладу часових рядів на три компоненти:

$$
Y_t = Trend_t + Seasonal_t + Residual_t
$$

Де:

* **Trend** — повільні довгострокові зміни
* **Seasonal** — регулярні циклічні коливання (тижні, місяці, дні тижня…)
* **Residual** — шум, помилки, аномалії та все, що не пояснено трендом/сезонністю

STL сам по собі **нічого статистично не перевіряє**, але дозволяє **побачити**, чи є в ряді тренд та сезонність.


## **resid**

За результатами STL-розкладу коефіцієнт варіації залишкової компоненти становить 1.62%. Це вказує на надзвичайно низький рівень необґрунтованих коливань. Практично вся варіація часового ряду пояснюється трендом і сезонністю, шум мінімальний.
